In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 🏠 House Prices Prediction — Advanced Regression Workflow

Author: *Jane Zhang*  
Date: *October 2025*

This notebook demonstrates a complete machine learning workflow for predicting house prices.  
It includes exploratory data analysis (EDA), feature engineering, preprocessing, and training advanced regression models such as **Linear Regression, Random Forest, XGBoost, and LightGBM**.  

The workflow highlights important features affecting house prices and provides evaluation metrics like RMSE and R².  
This notebook is designed to showcase professional Python and machine learning skills for data science portfolios.



---

### 📘 Table of Contents
1. [Introduction](#introduction)
2. [Data Overview](#data-overview)
3. [Exploratory Data Analysis (EDA)](#eda)
4. [Data Preprocessing](#data-preprocessing)
5. [Feature Engineering](#feature-engineering)
6. [Model Training and Evaluation](#model-training)
7. [Results and Discussion](#results)
8. [Conclusion](#conclusion)
9. [References](#references)

---

## 1. Introduction <a id="introduction"></a>

The goal of this project is to **predict house prices** based on various features such as location, size, and condition.  
We will explore the dataset, perform preprocessing and feature engineering, and train multiple regression models, including ensemble methods, to achieve strong predictive performance.

**Objectives:**
- Explore and clean the data  
- Engineer relevant features  
- Train multiple regression models (Linear, Random Forest, XGBoost, LightGBM)  
- Evaluate models using RMSE and R²  
- Interpret results and identify key factors affecting house prices  

---

## 2. Data Overview <a id="data-overview"></a>

We will use the **Kaggle House Prices dataset (train.csv)**:  
👉 [House Prices - Advanced Regression Techniques](https://www.kaggle.com/c/house-prices-advanced-regression-techniques/data)


In [ ]:
# === Import libraries ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb

# Plotting style
sns.set(style="whitegrid", palette="pastel", font_scale=1.1)

# === Load dataset ===
data = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
data.head()


---

## 3. Exploratory Data Analysis (EDA) <a id="eda"></a>

EDA helps us understand the data distribution, correlations, and potential feature issues such as missing values or outliers.



In [ ]:
# Dataset info
print("Dataset shape:", data.shape)
data.info()

# Check missing values
missing = data.isnull().sum().sort_values(ascending=False)
missing[missing > 0]


In [ ]:
# Target variable distribution
sns.histplot(data['SalePrice'], bins=50, kde=True, color='skyblue')
plt.title("Distribution of SalePrice")
plt.xlabel("SalePrice")
plt.ylabel("Frequency")
plt.show()


In [ ]:
# Correlation heatmap for top features

# Select only numeric features
numeric_data = data.select_dtypes(include=['int64', 'float64'])

# Compute correlation
corr = numeric_data.corr()

# Top 10 features correlated with SalePrice
top_corr_features = corr['SalePrice'].abs().sort_values(ascending=False).head(10).index

# Plot heatmap
plt.figure(figsize=(10,6))
sns.heatmap(numeric_data[top_corr_features].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Top Feature Correlations with SalePrice")
plt.show()


---

## 4. Data Preprocessing <a id="data-preprocessing"></a>

Steps include:
- Handling missing values  
- Encoding categorical features  
- Scaling numerical features  
- Splitting data into training and test sets  



In [ ]:
# Split features and target
X = data.drop(['Id', 'SalePrice'], axis=1)
y = data['SalePrice']

# Identify numerical and categorical features
num_features = X.select_dtypes(include=['int64', 'float64']).columns
cat_features = X.select_dtypes(include=['object']).columns

# Fill missing values
X[num_features] = X[num_features].fillna(X[num_features].median())
X[cat_features] = X[cat_features].fillna('Missing')

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


---

## 5. Feature Engineering <a id="feature-engineering"></a>

We create new features and encode categorical variables:
- One-hot encoding for categorical variables  
- Scaling for numerical variables  
- Optionally, combine features like total square footage



In [ ]:
# Column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ])


---

## 6. Model Training and Evaluation <a id="model-training"></a>

We will train multiple regression models and compare their performance:
- **Linear Regression** (baseline)  
- **Ridge & Lasso** (regularized linear models)  
- **Random Forest Regressor**  
- **XGBoost Regressor**  
- **LightGBM Regressor**

Evaluation metrics: **RMSE** and **R²**.



In [ ]:
# Helper function for model training and evaluation
def train_evaluate_model(model, X_train, X_test, y_train, y_test):
    pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('regressor', model)])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    print(f"{model.__class__.__name__} → RMSE: {rmse:.2f}, R²: {r2:.2f}")
    return pipeline, y_pred


In [ ]:
# Linear Regression
lr_model = LinearRegression()
lr_pipeline, y_pred_lr = train_evaluate_model(lr_model, X_train, X_test, y_train, y_test)

# Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=500, max_depth=10, random_state=42)
rf_pipeline, y_pred_rf = train_evaluate_model(rf_model, X_train, X_test, y_train, y_test)

# XGBoost Regressor
xgb_model = xgb.XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.05, random_state=42)
xgb_pipeline, y_pred_xgb = train_evaluate_model(xgb_model, X_train, X_test, y_train, y_test)

# LightGBM Regressor
lgb_model = lgb.LGBMRegressor(n_estimators=500, max_depth=10, learning_rate=0.05, random_state=42)
lgb_pipeline, y_pred_lgb = train_evaluate_model(lgb_model, X_train, X_test, y_train, y_test)


---

## 7. Results and Discussion <a id="results"></a>

**Observations:**
- Ensemble models (Random Forest, XGBoost, LightGBM) outperform simple Linear Regression.  
- Important features typically include **GrLivArea, OverallQual, TotalBsmtSF, GarageCars**.  
- RMSE values give a sense of average prediction error in dollars.  

**Next steps / improvements:**
- Hyperparameter tuning with `GridSearchCV` or `Optuna`  
- Advanced feature engineering (interaction terms, polynomial features)  
- Model interpretability using SHAP values  

---

## 8. Conclusion <a id="conclusion"></a>

This notebook demonstrated a **complete regression workflow**:
- Data exploration and cleaning  
- Feature engineering and preprocessing  
- Training advanced regression models  
- Evaluating model performance and identifying key factors affecting house prices  

The pipeline can be easily extended for Kaggle competitions or real-world price prediction tasks.

---

## 9. References <a id="references"></a>

- Kaggle Dataset: [House Prices - Advanced Regression Techniques](https://www.kaggle.com/c/house-prices-advanced-regression-techniques/data)  
- Scikit-learn: [https://scikit-learn.org](https://scikit-learn.org)  
- XGBoost Documentation: [https://xgboost.readthedocs.io](https://xgboost.readthedocs.io)  
- LightGBM Documentation: [https://lightgbm.readthedocs.io](https://lightgbm.readthedocs.io)
